# **Assign counties to model regions using a 5-percentage-point rule**

This notebook implements the county-to-region assignment rule.

For each county:

1. Calculate the share of candidate solar capacity in each model region.
2. Rank the model regions from largest to smallest capacity share.
3. Always retain the first-place region.
4. Retain the second-place region only when its capacity share is within **5 percentage points** of the first-place region.
5. Exclude third-place and lower-ranked regions.
6. Use only the actual candidate solar capacity associated with each retained county-region pair.

For example, if a county has:

- Region A: 34.4%
- Region B: 33.5%
- Region C: 32.1%

the county is retained in Regions A and B because the first- and second-place shares differ by only 0.9 percentage points.

The notebook also reports:

- how many counties are assigned to one region;
- how many counties are assigned to two regions;
- the resulting number of county-region assignments;
- retained and excluded candidate solar capacity;
- examples for Kern County and Jefferson County;
- quality-assurance checks; and
- final CSV outputs for downstream clustering.

## Step 1: Imports and notebook settings

In [61]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
pd.options.display.float_format = "{:,.4f}".format

## Step 2: Project paths

The notebook searches upward from the current working directory until it finds the `solar-county-analysis` repository.

The main input is the county-region capacity-share table generated by the earlier validation notebook.

In [62]:
def find_repo_root(start=None):
    """
    Find the solar-county-analysis repository root.

    The notebook can be run from the repo root or from its notebooks/ folder.
    """
    start = Path.cwd() if start is None else Path(start).resolve()

    candidates = [start] + list(start.parents)

    for path in candidates:
        if (
            (path / "notebooks").exists()
            and (path / "validation_outputs").exists()
        ):
            return path

    # Fallback to Lauren's known local repository path
    fallback = Path(
        "/Users/laurenvo/Documents/Github/solar-county-analysis"
    )

    if fallback.exists():
        return fallback

    raise FileNotFoundError(
        "Could not locate the solar-county-analysis repository. "
        "Run this notebook from the repository or notebooks/ folder."
    )


REPO_ROOT = find_repo_root()

VALIDATION_DIR = (
    REPO_ROOT
    / "validation_outputs"
    / "wecc_county_solar_5clusters"
)

INPUT_PATH = (
    VALIDATION_DIR
    / "all_county_region_capacity_shares.csv"
)

OUT_DIR = (
    REPO_ROOT
    / "validation_outputs"
    / "county_region_assignment_5pct_rule"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Input file:", INPUT_PATH)
print("Output folder:", OUT_DIR)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Required input file was not found:\n{INPUT_PATH}\n\n"
        "Run the multi-region county capacity-share section of "
        "validate_5cluster_county_capacity_threshold.ipynb first."
    )

Repository root: /Users/laurenvo/Documents/Github/solar-county-analysis
Input file: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/wecc_county_solar_5clusters/all_county_region_capacity_shares.csv
Output folder: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule


## Step 3: Load and validate the county-region capacity-share table

The expected input contains one row for each observed county-region pair.

Important fields include:

- `county_group`: county FIPS code;
- `county_name_full`: readable county name;
- `model_region`: model region;
- `capacity_mw`: candidate solar capacity in that county-region pair;
- `county_total_capacity_mw`: total candidate solar capacity across all regions for the county;
- `capacity_share_of_county`: the county-region capacity divided by total county capacity.

In [63]:
county_region = pd.read_csv(
    INPUT_PATH,
    dtype={"county_group": str},
)

county_region["county_group"] = (
    county_region["county_group"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

required_columns = [
    "county_group",
    "model_region",
    "capacity_mw",
    "county_total_capacity_mw",
    "capacity_share_of_county",
]

missing_columns = [
    column
    for column in required_columns
    if column not in county_region.columns
]

if missing_columns:
    raise ValueError(
        "The input table is missing required columns: "
        + ", ".join(missing_columns)
    )

numeric_columns = [
    "capacity_mw",
    "county_total_capacity_mw",
    "capacity_share_of_county",
]

for column in numeric_columns:
    county_region[column] = pd.to_numeric(
        county_region[column],
        errors="coerce",
    )

if county_region[numeric_columns].isna().any().any():
    bad_counts = (
        county_region[numeric_columns]
        .isna()
        .sum()
    )

    raise ValueError(
        "Some required numeric values could not be parsed:\n"
        f"{bad_counts}"
    )

print("Rows:", len(county_region))
print(
    "Unique counties:",
    county_region["county_group"].nunique(),
)
print(
    "Unique county-region pairs:",
    county_region[
        ["county_group", "model_region"]
    ].drop_duplicates().shape[0],
)
print(
    "Model regions:",
    county_region["model_region"].nunique(),
)

display(county_region.head())

Rows: 739
Unique counties: 419
Unique county-region pairs: 739
Model regions: 34


,county_group,model_region,candidate_rows,capacity_mw,county_total_candidate_rows,county_total_capacity_mw,n_model_regions,capacity_share_of_county,row_share_of_county,county_name_full,state_abbrev
0,04001,AZ2,82,"7,896.7906",1720,"211,973.6757",2,0.0373,0.0477,"Apache County, AZ",AZ
1,04001,AZ3,1638,"204,076.8851",1720,"211,973.6757",2,0.9627,0.9523,"Apache County, AZ",AZ
2,04003,AZ2,17,322.0047,872,"87,336.8009",2,0.0037,0.0195,"Cochise County, AZ",AZ
3,04003,AZ4,855,"87,014.7961",872,"87,336.8009",2,0.9963,0.9805,"Cochise County, AZ",AZ
4,04005,AZ1,1214,"143,152.6994",2078,"239,935.4394",5,0.5966,0.5842,"Coconino County, AZ",AZ


## Step 4: Recalculate and validate capacity shares

The capacity-share fields are recalculated from `capacity_mw` rather than accepted without checking.

For each county:

\[
\text{capacity share}_{c,r}
=
\frac{\text{capacity MW}_{c,r}}
{\sum_r \text{capacity MW}_{c,r}}
\]

The shares across all regions for a county should sum to approximately 100%.

In [64]:
# Aggregate defensively in case the input contains duplicate county-region rows.
county_region_clean = (
    county_region
    .groupby(
        ["county_group", "model_region"],
        as_index=False,
        dropna=False,
    )
    .agg(
        capacity_mw=("capacity_mw", "sum"),
        candidate_rows=(
            "candidate_rows",
            "sum",
        ) if "candidate_rows" in county_region.columns
        else ("county_group", "size"),
        county_name_full=(
            "county_name_full",
            "first",
        ) if "county_name_full" in county_region.columns
        else ("county_group", "first"),
        state_abbrev=(
            "state_abbrev",
            "first",
        ) if "state_abbrev" in county_region.columns
        else ("county_group", "first"),
    )
)

county_totals = (
    county_region_clean
    .groupby("county_group", as_index=False)
    .agg(
        county_total_capacity_mw=("capacity_mw", "sum"),
        county_total_candidate_rows=("candidate_rows", "sum"),
        n_original_model_regions=("model_region", "nunique"),
    )
)

county_region_clean = county_region_clean.merge(
    county_totals,
    on="county_group",
    how="left",
)

county_region_clean["capacity_share_of_county"] = (
    county_region_clean["capacity_mw"]
    / county_region_clean["county_total_capacity_mw"]
)

county_region_clean["capacity_share_pct"] = (
    county_region_clean["capacity_share_of_county"] * 100
)

share_check = (
    county_region_clean
    .groupby("county_group", as_index=False)
    .agg(
        capacity_share_sum=("capacity_share_of_county", "sum"),
        capacity_mw_sum=("capacity_mw", "sum"),
        stated_county_total_capacity_mw=(
            "county_total_capacity_mw",
            "first",
        ),
    )
)

share_check["share_sum_error"] = (
    share_check["capacity_share_sum"] - 1
).abs()

share_check["capacity_total_error_mw"] = (
    share_check["capacity_mw_sum"]
    - share_check["stated_county_total_capacity_mw"]
).abs()

print(
    "Counties:",
    share_check["county_group"].nunique(),
)
print(
    "Maximum capacity-share sum error:",
    share_check["share_sum_error"].max(),
)
print(
    "Counties whose shares do not sum to 1 within tolerance:",
    (share_check["share_sum_error"] > 1e-8).sum(),
)

display(
    county_region_clean
    .sort_values(
        ["county_group", "capacity_share_of_county"],
        ascending=[True, False],
    )
    .head(20)
)

Counties: 419
Maximum capacity-share sum error: 2.220446049250313e-16
Counties whose shares do not sum to 1 within tolerance: 0


,county_group,model_region,capacity_mw,candidate_rows,county_name_full,state_abbrev,county_total_capacity_mw,county_total_candidate_rows,n_original_model_regions,capacity_share_of_county,capacity_share_pct
1,04001,AZ3,"204,076.8851",1638,"Apache County, AZ",AZ,"211,973.6757",1720,2,0.9627,96.2746
0,04001,AZ2,"7,896.7906",82,"Apache County, AZ",AZ,"211,973.6757",1720,2,0.0373,3.7254
3,04003,AZ4,"87,014.7961",855,"Cochise County, AZ",AZ,"87,336.8009",872,2,0.9963,99.6313
2,04003,AZ2,322.0047,17,"Cochise County, AZ",AZ,"87,336.8009",872,2,0.0037,0.3687
4,04005,AZ1,"143,152.6994",1214,"Coconino County, AZ",AZ,"239,935.4394",2078,5,0.5966,59.6630
6,04005,AZ3,"67,589.5978",526,"Coconino County, AZ",AZ,"239,935.4394",2078,5,0.2817,28.1699
5,04005,AZ2,"17,330.1922",225,"Coconino County, AZ",AZ,"239,935.4394",2078,5,0.0722,7.2229
8,04005,NV2,"11,852.8249",112,"Coconino County, AZ",AZ,"239,935.4394",2078,5,0.0494,4.9400
7,04005,AZ4,10.1250,1,"Coconino County, AZ",AZ,"239,935.4394",2078,5,0.0000,0.0042
9,04007,AZ2,"25,585.4229",304,"Gila County, AZ",AZ,"25,990.6717",309,3,0.9844,98.4408


## Step 5: Rank model regions within each county

Regions are ranked separately for each county using candidate solar capacity.

- Rank 1 is the county's dominant region.
- Rank 2 is the county's second-largest region.
- Rank 3 and below will not be retained under proposed rule.

In [65]:
ranked = (
    county_region_clean
    .sort_values(
        [
            "county_group",
            "capacity_share_of_county",
            "capacity_mw",
            "model_region",
        ],
        ascending=[True, False, False, True],
    )
    .copy()
)

ranked["region_rank_within_county"] = (
    ranked
    .groupby("county_group")
    .cumcount()
    + 1
)

top_two = (
    ranked[
        ranked["region_rank_within_county"].isin([1, 2])
    ]
    .pivot(
        index="county_group",
        columns="region_rank_within_county",
        values=[
            "model_region",
            "capacity_share_of_county",
            "capacity_share_pct",
            "capacity_mw",
        ],
    )
)

# Flatten the pivoted column names.
top_two.columns = [
    f"{name}_rank_{int(rank)}"
    for name, rank in top_two.columns
]

top_two = top_two.reset_index()

# Counties with only one model region will not have rank-2 values.
for column in [
    "model_region_rank_2",
    "capacity_share_of_county_rank_2",
    "capacity_share_pct_rank_2",
    "capacity_mw_rank_2",
]:
    if column not in top_two.columns:
        top_two[column] = np.nan

top_two["top_second_gap_share"] = (
    top_two["capacity_share_of_county_rank_1"]
    - top_two["capacity_share_of_county_rank_2"]
)

top_two["top_second_gap_pct"] = (
    top_two["capacity_share_pct_rank_1"]
    - top_two["capacity_share_pct_rank_2"]
)

top_two["has_second_region"] = (
    top_two["model_region_rank_2"].notna()
)

display(
    top_two
    .sort_values("top_second_gap_pct")
    .head(20)
)

,county_group,model_region_rank_1,model_region_rank_2,capacity_share_of_county_rank_1,capacity_share_of_county_rank_2,capacity_share_pct_rank_1,capacity_share_pct_rank_2,capacity_mw_rank_1,capacity_mw_rank_2,top_second_gap_share,top_second_gap_pct,has_second_region
246,35003,AZ3,NM1,0.5015,0.4936,50.1451,49.3561,"29,174.4858","28,715.4496",0.0079,0.7890,True
29,06029,NV2,CA2,0.3438,0.3351,34.3784,33.5134,"23,654.5307","23,059.3864",0.0086,0.8650,True
158,16053,ID1,ID3,0.5095,0.4905,50.9535,49.0465,"5,742.9400","5,528.0015",0.0191,1.9070,True
405,56019,WY3,WY2,0.5097,0.4903,50.9722,49.0278,"37,513.6218","36,082.6432",0.0194,1.9444,True
171,16079,ID2,WA4,0.5102,0.4898,51.0243,48.9757,237.4077,227.8758,0.0205,2.0486,True
47,06065,CA2,AZ1,0.5057,0.4846,50.5734,48.4627,"11,203.2004","10,735.6439",0.0211,2.1106,True
364,53015,WA1,OR1,0.4788,0.4555,47.8770,45.5491,"3,659.0921","3,481.1803",0.0233,2.3279,True
247,35006,AZ3,NM1,0.5121,0.4879,51.2109,48.7891,"36,591.2441","34,860.8191",0.0242,2.4218,True
177,30003,MT3,WY2,0.5233,0.4767,52.3312,47.6688,"52,777.0885","48,074.9384",0.0466,4.6624,True
355,49055,UT1,WY1,0.5290,0.4710,52.8961,47.1039,"7,404.4011","6,593.6083",0.0579,5.7922,True


## Step 6: Apply the 5-percentage-point assignment rule

This notebook interprets "within 5%" as **within 5 percentage points**.

The rule is:

- always keep rank 1;
- keep rank 2 when:

\[
\text{rank-1 share} - \text{rank-2 share}
\leq 5 \text{ percentage points}
\]

Only the first- and second-place regions can be retained.

In [66]:
TOP_TWO_GAP_THRESHOLD_PCT = 5.0

ranked = ranked.merge(
    top_two[
        [
            "county_group",
            "model_region_rank_1",
            "model_region_rank_2",
            "capacity_share_pct_rank_1",
            "capacity_share_pct_rank_2",
            "top_second_gap_pct",
            "has_second_region",
        ]
    ],
    on="county_group",
    how="left",
)

ranked["keep_under_5pct_rule"] = (
    ranked["region_rank_within_county"].eq(1)
    |
    (
        ranked["region_rank_within_county"].eq(2)
        & ranked["has_second_region"]
        & ranked["top_second_gap_pct"].le(
            TOP_TWO_GAP_THRESHOLD_PCT
        )
    )
)

ranked["assignment_reason"] = np.select(
    [
        ranked["region_rank_within_county"].eq(1),
        (
            ranked["region_rank_within_county"].eq(2)
            & ranked["top_second_gap_pct"].le(
                TOP_TWO_GAP_THRESHOLD_PCT
            )
        ),
    ],
    [
        "dominant_region",
        "second_region_within_5_percentage_points",
    ],
    default="excluded",
)

assignments = (
    ranked[
        ranked["keep_under_5pct_rule"]
    ]
    .copy()
)

assignments["assigned_buildable_mw"] = (
    assignments["capacity_mw"]
)

assignments["assigned_capacity_share_of_county"] = (
    assignments["capacity_share_of_county"]
)

assignments["assigned_capacity_share_pct"] = (
    assignments["capacity_share_pct"]
)

print(
    "Retained county-region assignments:",
    len(assignments),
)
print(
    "Unique counties represented:",
    assignments["county_group"].nunique(),
)

display(
    assignments[
        [
            "county_group",
            "county_name_full",
            "state_abbrev",
            "model_region",
            "region_rank_within_county",
            "capacity_share_pct",
            "top_second_gap_pct",
            "assignment_reason",
            "assigned_buildable_mw",
        ]
    ]
    .sort_values(
        ["county_group", "region_rank_within_county"]
    )
    .head(30)
)

Retained county-region assignments: 428
Unique counties represented: 419


,county_group,county_name_full,state_abbrev,model_region,region_rank_within_county,capacity_share_pct,top_second_gap_pct,assignment_reason,assigned_buildable_mw
0,04001,"Apache County, AZ",AZ,AZ3,1,96.2746,92.5493,dominant_region,"204,076.8851"
2,04003,"Cochise County, AZ",AZ,AZ4,1,99.6313,99.2626,dominant_region,"87,014.7961"
4,04005,"Coconino County, AZ",AZ,AZ1,1,59.6630,31.4931,dominant_region,"143,152.6994"
9,04007,"Gila County, AZ",AZ,AZ2,1,98.4408,97.0599,dominant_region,"25,585.4229"
12,04009,"Graham County, AZ",AZ,AZ4,1,55.1201,10.2402,dominant_region,"33,612.7114"
14,04011,"Greenlee County, AZ",AZ,AZ4,1,100.0000,NaN,dominant_region,"4,337.8142"
15,04012,"La Paz County, AZ",AZ,AZ1,1,92.8246,85.6492,dominant_region,"36,432.4339"
17,04013,"Maricopa County, AZ",AZ,AZ2,1,93.0540,86.1081,dominant_region,"55,866.8124"
19,04015,"Mohave County, AZ",AZ,AZ1,1,65.6120,31.3184,dominant_region,"87,845.1580"
22,04017,"Navajo County, AZ",AZ,AZ3,1,93.8403,89.8085,dominant_region,"168,893.0424"


## Step 7: Count one-region and two-region counties

Every county should appear at least once.

A county appears twice only when the difference between its first- and second-place region shares is no more than 5 percentage points.

In [67]:
assignment_counts = (
    assignments
    .groupby("county_group", as_index=False)
    .agg(
        county_name_full=("county_name_full", "first"),
        state_abbrev=("state_abbrev", "first"),
        retained_region_count=("model_region", "nunique"),
        retained_capacity_mw=("assigned_buildable_mw", "sum"),
        county_total_capacity_mw=(
            "county_total_capacity_mw",
            "first",
        ),
        retained_capacity_share=(
            "assigned_capacity_share_of_county",
            "sum",
        ),
        top_second_gap_pct=("top_second_gap_pct", "first"),
    )
)

assignment_counts["retained_capacity_share_pct"] = (
    assignment_counts["retained_capacity_share"] * 100
)

assignment_counts["excluded_capacity_share"] = (
    1 - assignment_counts["retained_capacity_share"]
)

assignment_counts["excluded_capacity_share_pct"] = (
    assignment_counts["excluded_capacity_share"] * 100
)

assignment_counts["excluded_capacity_mw"] = (
    assignment_counts["county_total_capacity_mw"]
    - assignment_counts["retained_capacity_mw"]
)

one_region_counties = int(
    assignment_counts["retained_region_count"].eq(1).sum()
)

two_region_counties = int(
    assignment_counts["retained_region_count"].eq(2).sum()
)

total_counties = assignment_counts["county_group"].nunique()
total_assignments = len(assignments)

print("===== COUNTY-TO-REGION ASSIGNMENT SUMMARY =====")
print("Total counties:", total_counties)
print("Counties assigned to one region:", one_region_counties)
print("Counties assigned to two regions:", two_region_counties)
print("Total retained county-region assignments:", total_assignments)
print(
    "Expected assignment count:",
    total_counties + two_region_counties,
)

display(
    assignment_counts[
        [
            "retained_region_count",
        ]
    ]
    .value_counts()
    .rename("county_count")
    .reset_index()
    .sort_values("retained_region_count")
)

===== COUNTY-TO-REGION ASSIGNMENT SUMMARY =====
Total counties: 419
Counties assigned to one region: 410
Counties assigned to two regions: 9
Total retained county-region assignments: 428
Expected assignment count: 428


,retained_region_count,county_count
0,1,410
1,2,9


## Step 8: Create a readable one-row-per-county summary

This table shows:

- the dominant region;
- the second-place region;
- the percentage-point gap;
- whether the second region is retained;
- all retained regions and their capacity shares;
- retained and excluded candidate solar capacity.

In [68]:
def format_retained_region_split(group):
    group = group.sort_values(
        "region_rank_within_county"
    )

    return "; ".join(
        (
            f"{row.model_region}: "
            f"{row.assigned_capacity_share_pct:.1f}% "
            f"({row.assigned_buildable_mw:,.1f} MW)"
        )
        for row in group.itertuples()
    )


retained_region_strings = (
    assignments
    .groupby("county_group")
    .apply(format_retained_region_split)
    .rename("retained_region_capacity_split")
    .reset_index()
)

county_assignment_summary = (
    assignment_counts
    .merge(
        top_two[
            [
                "county_group",
                "model_region_rank_1",
                "model_region_rank_2",
                "capacity_share_pct_rank_1",
                "capacity_share_pct_rank_2",
                "capacity_mw_rank_1",
                "capacity_mw_rank_2",
                "top_second_gap_pct",
            ]
        ],
        on="county_group",
        how="left",
        suffixes=("", "_top_two"),
    )
    .merge(
        retained_region_strings,
        on="county_group",
        how="left",
    )
)

county_assignment_summary["second_region_retained"] = (
    county_assignment_summary[
        "retained_region_count"
    ].eq(2)
)

county_assignment_summary = (
    county_assignment_summary
    .sort_values(
        [
            "second_region_retained",
            "top_second_gap_pct",
            "county_total_capacity_mw",
        ],
        ascending=[False, True, False],
    )
)

display(
    county_assignment_summary[
        [
            "county_group",
            "county_name_full",
            "state_abbrev",
            "model_region_rank_1",
            "capacity_share_pct_rank_1",
            "model_region_rank_2",
            "capacity_share_pct_rank_2",
            "top_second_gap_pct",
            "second_region_retained",
            "retained_region_capacity_split",
            "county_total_capacity_mw",
            "retained_capacity_mw",
            "excluded_capacity_mw",
        ]
    ]
    .head(30)
)

,county_group,county_name_full,state_abbrev,model_region_rank_1,capacity_share_pct_rank_1,model_region_rank_2,capacity_share_pct_rank_2,top_second_gap_pct,second_region_retained,retained_region_capacity_split,county_total_capacity_mw,retained_capacity_mw,excluded_capacity_mw
246,35003,"Catron County, NM",NM,AZ3,50.1451,NM1,49.3561,0.7890,True,"AZ3: 50.1% (29,174.5 MW); NM1: 49.4% (28,715.4 MW)","58,180.1599","57,889.9354",290.2246
29,06029,"Kern County, CA",CA,NV2,34.3784,CA2,33.5134,0.8650,True,"NV2: 34.4% (23,654.5 MW); CA2: 33.5% (23,059.4 MW)","68,806.4374","46,713.9171","22,092.5204"
158,16053,"Jerome County, ID",ID,ID1,50.9535,ID3,49.0465,1.9070,True,"ID1: 51.0% (5,742.9 MW); ID3: 49.0% (5,528.0 MW)","11,270.9415","11,270.9415",0.0000
405,56019,"Johnson County, WY",WY,WY3,50.9722,WY2,49.0278,1.9444,True,"WY3: 51.0% (37,513.6 MW); WY2: 49.0% (36,082.6 MW)","73,596.2649","73,596.2649",0.0000
171,16079,"Shoshone County, ID",ID,ID2,51.0243,WA4,48.9757,2.0486,True,ID2: 51.0% (237.4 MW); WA4: 49.0% (227.9 MW),465.2835,465.2835,0.0000
47,06065,"Riverside County, CA",CA,CA2,50.5734,AZ1,48.4627,2.1106,True,"CA2: 50.6% (11,203.2 MW); AZ1: 48.5% (10,735.6 MW)","22,152.3624","21,938.8443",213.5181
364,53015,"Cowlitz County, WA",WA,WA1,47.8770,OR1,45.5491,2.3279,True,"WA1: 47.9% (3,659.1 MW); OR1: 45.5% (3,481.2 MW)","7,642.6968","7,140.2724",502.4244
247,35006,"Cibola County, NM",NM,AZ3,51.2109,NM1,48.7891,2.4218,True,"AZ3: 51.2% (36,591.2 MW); NM1: 48.8% (34,860.8 MW)","71,452.0632","71,452.0632",0.0000
177,30003,"Big Horn County, MT",MT,MT3,52.3312,WY2,47.6688,4.6624,True,"MT3: 52.3% (52,777.1 MW); WY2: 47.7% (48,074.9 MW)","100,852.0269","100,852.0269",0.0000
355,49055,"Wayne County, UT",UT,UT1,52.8961,WY1,47.1039,5.7922,False,"UT1: 52.9% (7,404.4 MW)","13,998.0094","7,404.4011","6,593.6083"


## Step 9: Inspect counties assigned to two regions

How many counties have first- and second-place region shares within 5 percentage points?

In [69]:
two_region_assignments = (
    county_assignment_summary[
        county_assignment_summary[
            "second_region_retained"
        ]
    ]
    .copy()
)

print(
    "Counties whose first- and second-place regions "
    "are within 5 percentage points:",
    len(two_region_assignments),
)

display(
    two_region_assignments[
        [
            "county_group",
            "county_name_full",
            "state_abbrev",
            "model_region_rank_1",
            "capacity_share_pct_rank_1",
            "model_region_rank_2",
            "capacity_share_pct_rank_2",
            "top_second_gap_pct",
            "retained_region_capacity_split",
            "county_total_capacity_mw",
            "retained_capacity_mw",
            "excluded_capacity_mw",
        ]
    ]
    .sort_values(
        [
            "top_second_gap_pct",
            "county_total_capacity_mw",
        ],
        ascending=[True, False],
    )
)

Counties whose first- and second-place regions are within 5 percentage points: 9


,county_group,county_name_full,state_abbrev,model_region_rank_1,capacity_share_pct_rank_1,model_region_rank_2,capacity_share_pct_rank_2,top_second_gap_pct,retained_region_capacity_split,county_total_capacity_mw,retained_capacity_mw,excluded_capacity_mw
246,35003,"Catron County, NM",NM,AZ3,50.1451,NM1,49.3561,0.7890,"AZ3: 50.1% (29,174.5 MW); NM1: 49.4% (28,715.4 MW)","58,180.1599","57,889.9354",290.2246
29,06029,"Kern County, CA",CA,NV2,34.3784,CA2,33.5134,0.8650,"NV2: 34.4% (23,654.5 MW); CA2: 33.5% (23,059.4 MW)","68,806.4374","46,713.9171","22,092.5204"
158,16053,"Jerome County, ID",ID,ID1,50.9535,ID3,49.0465,1.9070,"ID1: 51.0% (5,742.9 MW); ID3: 49.0% (5,528.0 MW)","11,270.9415","11,270.9415",0.0000
405,56019,"Johnson County, WY",WY,WY3,50.9722,WY2,49.0278,1.9444,"WY3: 51.0% (37,513.6 MW); WY2: 49.0% (36,082.6 MW)","73,596.2649","73,596.2649",0.0000
171,16079,"Shoshone County, ID",ID,ID2,51.0243,WA4,48.9757,2.0486,ID2: 51.0% (237.4 MW); WA4: 49.0% (227.9 MW),465.2835,465.2835,0.0000
47,06065,"Riverside County, CA",CA,CA2,50.5734,AZ1,48.4627,2.1106,"CA2: 50.6% (11,203.2 MW); AZ1: 48.5% (10,735.6 MW)","22,152.3624","21,938.8443",213.5181
364,53015,"Cowlitz County, WA",WA,WA1,47.8770,OR1,45.5491,2.3279,"WA1: 47.9% (3,659.1 MW); OR1: 45.5% (3,481.2 MW)","7,642.6968","7,140.2724",502.4244
247,35006,"Cibola County, NM",NM,AZ3,51.2109,NM1,48.7891,2.4218,"AZ3: 51.2% (36,591.2 MW); NM1: 48.8% (34,860.8 MW)","71,452.0632","71,452.0632",0.0000
177,30003,"Big Horn County, MT",MT,MT3,52.3312,WY2,47.6688,4.6624,"MT3: 52.3% (52,777.1 MW); WY2: 47.7% (48,074.9 MW)","100,852.0269","100,852.0269",0.0000


## Step 10: Inspect Kern County

Kern County, California, county FIPS `06029`.

The expected region shares were approximately:

- NV2: 34.4%
- CA2: 33.5%
- CA4: 32.1%

Because the difference between NV2 and CA2 is less than 5 percentage points, the rule should retain both NV2 and CA2.

In [70]:
KERN_FIPS = "06029"

kern_all_regions = (
    ranked[
        ranked["county_group"].eq(KERN_FIPS)
    ]
    [
        [
            "county_group",
            "county_name_full",
            "model_region",
            "region_rank_within_county",
            "capacity_mw",
            "county_total_capacity_mw",
            "capacity_share_pct",
            "top_second_gap_pct",
            "keep_under_5pct_rule",
            "assignment_reason",
        ]
    ]
    .sort_values("region_rank_within_county")
)

print("===== KERN COUNTY CHECK =====")

if kern_all_regions.empty:
    print(
        "Kern County was not found in the input table."
    )
else:
    display(kern_all_regions)

    kern_retained = kern_all_regions[
        kern_all_regions["keep_under_5pct_rule"]
    ]

    print(
        "Retained regions:",
        ", ".join(kern_retained["model_region"]),
    )
    print(
        "Retained capacity share:",
        f"{kern_retained['capacity_share_pct'].sum():.1f}%",
    )
    print(
        "Retained capacity:",
        f"{kern_retained['capacity_mw'].sum():,.1f} MW",
    )

===== KERN COUNTY CHECK =====


,county_group,county_name_full,model_region,region_rank_within_county,capacity_mw,county_total_capacity_mw,capacity_share_pct,top_second_gap_pct,keep_under_5pct_rule,assignment_reason
55,06029,"Kern County, CA",NV2,1,"23,654.5307","68,806.4374",34.3784,0.8650,True,dominant_region
56,06029,"Kern County, CA",CA2,2,"23,059.3864","68,806.4374",33.5134,0.8650,True,second_region_within_5_percentage_points
57,06029,"Kern County, CA",CA4,3,"22,092.5204","68,806.4374",32.1082,0.8650,False,excluded


Retained regions: NV2, CA2
Retained capacity share: 67.9%
Retained capacity: 46,713.9 MW


In [71]:
print("All counties meeting the two-region rule:")
display(
    two_region_assignments[
        [
            "county_group",
            "county_name_full",
            "model_region_rank_1",
            "capacity_share_pct_rank_1",
            "model_region_rank_2",
            "capacity_share_pct_rank_2",
            "top_second_gap_pct",
        ]
    ].sort_values("top_second_gap_pct")
)

All counties meeting the two-region rule:


,county_group,county_name_full,model_region_rank_1,capacity_share_pct_rank_1,model_region_rank_2,capacity_share_pct_rank_2,top_second_gap_pct
246,35003,"Catron County, NM",AZ3,50.1451,NM1,49.3561,0.7890
29,06029,"Kern County, CA",NV2,34.3784,CA2,33.5134,0.8650
158,16053,"Jerome County, ID",ID1,50.9535,ID3,49.0465,1.9070
405,56019,"Johnson County, WY",WY3,50.9722,WY2,49.0278,1.9444
171,16079,"Shoshone County, ID",ID2,51.0243,WA4,48.9757,2.0486
47,06065,"Riverside County, CA",CA2,50.5734,AZ1,48.4627,2.1106
364,53015,"Cowlitz County, WA",WA1,47.8770,OR1,45.5491,2.3279
247,35006,"Cibola County, NM",AZ3,51.2109,NM1,48.7891,2.4218
177,30003,"Big Horn County, MT",MT3,52.3312,WY2,47.6688,4.6624


## Step 11: Inspect Jefferson County and MT2 capacity

If only 37.9% of Jefferson County's total candidate capacity falls in MT2, the Jefferson County–MT2 row should use only that proportional capacity.

The notebook uses the already aggregated county-region `capacity_mw` value directly. This is equivalent to:

**Assigned buildable MW = County total capacity × MT2 capacity share**

and avoids assigning the county's full capacity to every retained region.

In [72]:
jefferson_rows = (
    ranked[
        ranked["county_name_full"]
        .astype(str)
        .str.contains(
            "Jefferson",
            case=False,
            na=False,
        )
    ]
    [
        [
            "county_group",
            "county_name_full",
            "state_abbrev",
            "model_region",
            "region_rank_within_county",
            "county_total_capacity_mw",
            "capacity_share_pct",
            "capacity_mw",
            "keep_under_5pct_rule",
            "assignment_reason",
        ]
    ]
    .sort_values(
        [
            "county_name_full",
            "region_rank_within_county",
        ]
    )
)

print("===== JEFFERSON COUNTY ROWS =====")
display(jefferson_rows)

jefferson_mt2 = jefferson_rows[
    jefferson_rows["model_region"].eq("MT2")
].copy()

if not jefferson_mt2.empty:
    jefferson_mt2["calculated_from_share_mw"] = (
        jefferson_mt2["county_total_capacity_mw"]
        * jefferson_mt2["capacity_share_pct"]
        / 100
    )

    jefferson_mt2["difference_from_capacity_mw"] = (
        jefferson_mt2["capacity_mw"]
        - jefferson_mt2["calculated_from_share_mw"]
    )

    print("===== JEFFERSON COUNTY–MT2 CHECK =====")

    display(
        jefferson_mt2[
            [
                "county_group",
                "county_name_full",
                "county_total_capacity_mw",
                "capacity_share_pct",
                "calculated_from_share_mw",
                "capacity_mw",
                "difference_from_capacity_mw",
            ]
        ]
    )
else:
    print(
        "No Jefferson County–MT2 row was found."
    )

===== JEFFERSON COUNTY ROWS =====


,county_group,county_name_full,state_abbrev,model_region,region_rank_within_county,county_total_capacity_mw,capacity_share_pct,capacity_mw,keep_under_5pct_rule,assignment_reason
169,08059,"Jefferson County, CO",CO,CO1,1,808.8231,100.0000,808.8231,True,dominant_region
266,16051,"Jefferson County, ID",ID,ID3,1,"20,152.3290",100.0000,"20,152.3290",True,dominant_region
328,30043,"Jefferson County, MT",MT,MT2,1,"13,190.9261",37.9257,"5,002.7490",True,dominant_region
329,30043,"Jefferson County, MT",MT,MT1,2,"13,190.9261",27.9265,"3,683.7607",False,excluded
330,30043,"Jefferson County, MT",MT,ID3,3,"13,190.9261",27.8864,"3,678.4787",False,excluded
331,30043,"Jefferson County, MT",MT,MT3,4,"13,190.9261",6.2614,825.9377,False,excluded
481,41031,"Jefferson County, OR",OR,WA2,1,"19,925.0366",73.3334,"14,611.7085",True,dominant_region
482,41031,"Jefferson County, OR",OR,OR1,2,"19,925.0366",26.6666,"5,313.3281",False,excluded
650,53031,"Jefferson County, WA",WA,WA1,1,"8,077.4465",100.0000,"8,077.4465",True,dominant_region


===== JEFFERSON COUNTY–MT2 CHECK =====


,county_group,county_name_full,county_total_capacity_mw,capacity_share_pct,calculated_from_share_mw,capacity_mw,difference_from_capacity_mw
328,30043,"Jefferson County, MT","13,190.9261",37.9257,"5,002.7490","5,002.7490",0.0000


## Step 12: Measure retained and excluded capacity

The rule keeps only:

- the first-place region for every county; and
- the second-place region when it is within 5 percentage points.

Therefore, capacity in third-place and lower regions is excluded. Capacity in the second-place region is also excluded when its gap from first place exceeds 5 percentage points.

This section quantifies the effect instead of silently discarding that capacity.

In [73]:
original_total_capacity_mw = (
    county_region_clean["capacity_mw"].sum()
)

retained_total_capacity_mw = (
    assignments["assigned_buildable_mw"].sum()
)

excluded_total_capacity_mw = (
    original_total_capacity_mw
    - retained_total_capacity_mw
)

retained_total_capacity_pct = (
    retained_total_capacity_mw
    / original_total_capacity_mw
    * 100
)

excluded_total_capacity_pct = (
    excluded_total_capacity_mw
    / original_total_capacity_mw
    * 100
)

capacity_summary = pd.DataFrame(
    [
        {
            "measure": "original_candidate_capacity_mw",
            "value": original_total_capacity_mw,
        },
        {
            "measure": "retained_candidate_capacity_mw",
            "value": retained_total_capacity_mw,
        },
        {
            "measure": "excluded_candidate_capacity_mw",
            "value": excluded_total_capacity_mw,
        },
        {
            "measure": "retained_candidate_capacity_pct",
            "value": retained_total_capacity_pct,
        },
        {
            "measure": "excluded_candidate_capacity_pct",
            "value": excluded_total_capacity_pct,
        },
    ]
)

print("===== CAPACITY RETENTION SUMMARY =====")
print(
    "Original candidate capacity:",
    f"{original_total_capacity_mw:,.2f} MW",
)
print(
    "Retained candidate capacity:",
    f"{retained_total_capacity_mw:,.2f} MW",
    f"({retained_total_capacity_pct:.2f}%)",
)
print(
    "Excluded candidate capacity:",
    f"{excluded_total_capacity_mw:,.2f} MW",
    f"({excluded_total_capacity_pct:.2f}%)",
)

display(capacity_summary)

===== CAPACITY RETENTION SUMMARY =====
Original candidate capacity: 11,715,680.05 MW
Retained candidate capacity: 9,957,637.09 MW (84.99%)
Excluded candidate capacity: 1,758,042.96 MW (15.01%)


,measure,value
0,original_candidate_capacity_mw,"11,715,680.0535"
1,retained_candidate_capacity_mw,"9,957,637.0916"
2,excluded_candidate_capacity_mw,"1,758,042.9618"
3,retained_candidate_capacity_pct,84.9941
4,excluded_candidate_capacity_pct,15.0059


## Step 13: Quality-assurance checks

The checks below verify that:

1. every original county is represented;
2. each county has either one or two retained assignments;
3. no third-place or lower region is retained;
4. second-place assignments satisfy the 5-percentage-point rule;
5. retained buildable capacity equals the original county-region capacity;
6. county retained capacity never exceeds county total capacity; and
7. there are no duplicate retained county-region pairs.

In [74]:
qa_results = []

# 1. Every county is represented
original_county_count = county_region_clean["county_group"].nunique()
retained_county_count = assignments["county_group"].nunique()

qa_results.append(
    {
        "check": "Every county is represented",
        "passed": original_county_count == retained_county_count,
        "details": (
            f"{retained_county_count} retained of "
            f"{original_county_count} original counties"
        ),
    }
)

# 2. Each county has one or two retained assignments
valid_assignment_counts = (
    assignment_counts["retained_region_count"]
    .isin([1, 2])
    .all()
)

qa_results.append(
    {
        "check": "Each county has one or two assignments",
        "passed": bool(valid_assignment_counts),
        "details": str(
            assignment_counts["retained_region_count"]
            .value_counts()
            .sort_index()
            .to_dict()
        ),
    }
)

# 3. No third-place or lower region is retained
no_rank_above_two = (
    assignments["region_rank_within_county"]
    .le(2)
    .all()
)

qa_results.append(
    {
        "check": "No rank 3+ region is retained",
        "passed": bool(no_rank_above_two),
        "details": (
            f"Maximum retained rank = "
            f"{assignments['region_rank_within_county'].max()}"
        ),
    }
)

# 4. Every retained second-place region satisfies the 5-point rule
retained_second_rows = assignments[
    assignments["region_rank_within_county"].eq(2)
]

second_rows_meet_rule = (
    retained_second_rows["top_second_gap_pct"]
    .le(TOP_TWO_GAP_THRESHOLD_PCT + 1e-10)
    .all()
)

qa_results.append(
    {
        "check": "All retained second regions satisfy 5-point rule",
        "passed": bool(second_rows_meet_rule),
        "details": (
            f"{len(retained_second_rows)} retained "
            "second-place rows"
        ),
    }
)

# 5. Assigned buildable MW equals actual county-region capacity
capacity_values_match = np.allclose(
    assignments["assigned_buildable_mw"],
    assignments["capacity_mw"],
    rtol=0,
    atol=1e-8,
)

qa_results.append(
    {
        "check": "Assigned buildable MW equals county-region MW",
        "passed": bool(capacity_values_match),
        "details": (
            "assigned_buildable_mw uses each "
            "county-region pair's actual capacity_mw"
        ),
    }
)

# 6. Retained county capacity does not exceed total county capacity
retained_not_above_total = (
    assignment_counts["retained_capacity_mw"]
    .le(
        assignment_counts["county_total_capacity_mw"]
        + 1e-8
    )
    .all()
)

max_capacity_excess_mw = (
    assignment_counts["retained_capacity_mw"]
    - assignment_counts["county_total_capacity_mw"]
).max()

qa_results.append(
    {
        "check": "Retained county capacity does not exceed total",
        "passed": bool(retained_not_above_total),
        "details": (
            f"Maximum excess = "
            f"{max_capacity_excess_mw:,.8f} MW"
        ),
    }
)

# 7. No duplicate county-region assignments
duplicate_count = (
    assignments[
        ["county_group", "model_region"]
    ]
    .duplicated()
    .sum()
)

qa_results.append(
    {
        "check": "No duplicate county-region assignments",
        "passed": duplicate_count == 0,
        "details": f"{duplicate_count} duplicates",
    }
)

# Display results
qa_table = pd.DataFrame(qa_results)

display(qa_table)

if not qa_table["passed"].all():
    failed = qa_table[~qa_table["passed"]]

    raise AssertionError(
        "One or more QA checks failed:\n"
        + failed.to_string(index=False)
    )

print("PASS: All assignment QA checks passed.")

,check,passed,details
0,Every county is represented,True,419 retained of 419 original counties
1,Each county has one or two assignments,True,"{1: 410, 2: 9}"
2,No rank 3+ region is retained,True,Maximum retained rank = 2
3,All retained second regions satisfy 5-point rule,True,9 retained second-place rows
4,Assigned buildable MW equals county-region MW,True,assigned_buildable_mw uses each county-region pair's actual capacity_mw
5,Retained county capacity does not exceed total,True,Maximum excess = 0.00000000 MW
6,No duplicate county-region assignments,True,0 duplicates


PASS: All assignment QA checks passed.


## Step 14: Sensitivity check for alternative gap thresholds

Jenny requested a 5-percentage-point threshold. This sensitivity table shows how the number of two-region counties and retained capacity would change under nearby thresholds.

This does not change the final assignment; it is only a diagnostic.

In [75]:
def summarize_gap_threshold(threshold_pct):
    keep = (
        ranked["region_rank_within_county"].eq(1)
        |
        (
            ranked["region_rank_within_county"].eq(2)
            & ranked["has_second_region"]
            & ranked["top_second_gap_pct"].le(
                threshold_pct
            )
        )
    )

    selected = ranked[keep].copy()

    per_county = (
        selected
        .groupby("county_group")
        .agg(
            retained_regions=("model_region", "nunique"),
            retained_capacity_mw=("capacity_mw", "sum"),
        )
    )

    return {
        "gap_threshold_pct_points": threshold_pct,
        "counties": per_county.shape[0],
        "one_region_counties": int(
            per_county["retained_regions"].eq(1).sum()
        ),
        "two_region_counties": int(
            per_county["retained_regions"].eq(2).sum()
        ),
        "county_region_assignments": len(selected),
        "retained_capacity_mw": (
            selected["capacity_mw"].sum()
        ),
        "retained_capacity_pct": (
            selected["capacity_mw"].sum()
            / original_total_capacity_mw
            * 100
        ),
    }


threshold_sensitivity = pd.DataFrame(
    [
        summarize_gap_threshold(threshold)
        for threshold in [0, 1, 2.5, 5, 7.5, 10]
    ]
)

display(threshold_sensitivity)

,gap_threshold_pct_points,counties,one_region_counties,two_region_counties,county_region_assignments,retained_capacity_mw,retained_capacity_pct
0,0.0000,419,419,0,419,"9,766,871.1535",83.3658
1,1.0000,419,417,2,421,"9,818,645.9895",83.8077
2,2.5000,419,411,8,427,"9,909,562.1532",84.5838
3,5.0000,419,410,9,428,"9,957,637.0916",84.9941
4,7.5000,419,406,13,432,"10,028,899.1494",85.6024
5,10.0000,419,403,16,435,"10,048,476.5994",85.7695


## Step 15: Prepare final output columns

The main assignment table contains one row per retained county-region pair.

`assigned_buildable_mw` is the capacity that should be associated with that specific county-region assignment. It is not the county's full capacity unless 100% of the county's candidate solar capacity belongs to that region.

In [76]:
final_assignment_columns = [
    "county_group",
    "county_name_full",
    "state_abbrev",
    "model_region",
    "region_rank_within_county",
    "assignment_reason",
    "capacity_mw",
    "county_total_capacity_mw",
    "capacity_share_of_county",
    "capacity_share_pct",
    "assigned_buildable_mw",
    "top_second_gap_pct",
    "n_original_model_regions",
    "candidate_rows",
    "county_total_candidate_rows",
]

final_assignment_columns = [
    column
    for column in final_assignment_columns
    if column in assignments.columns
]

final_assignments = (
    assignments[
        final_assignment_columns
    ]
    .sort_values(
        [
            "county_group",
            "region_rank_within_county",
        ]
    )
    .reset_index(drop=True)
)

display(final_assignments.head(30))

print("Final rows:", len(final_assignments))
print(
    "Final counties:",
    final_assignments["county_group"].nunique(),
)
print(
    "Final assigned buildable MW:",
    f"{final_assignments['assigned_buildable_mw'].sum():,.2f}",
)

,county_group,county_name_full,state_abbrev,model_region,region_rank_within_county,assignment_reason,capacity_mw,county_total_capacity_mw,capacity_share_of_county,capacity_share_pct,assigned_buildable_mw,top_second_gap_pct,n_original_model_regions,candidate_rows,county_total_candidate_rows
0,04001,"Apache County, AZ",AZ,AZ3,1,dominant_region,"204,076.8851","211,973.6757",0.9627,96.2746,"204,076.8851",92.5493,2,1638,1720
1,04003,"Cochise County, AZ",AZ,AZ4,1,dominant_region,"87,014.7961","87,336.8009",0.9963,99.6313,"87,014.7961",99.2626,2,855,872
2,04005,"Coconino County, AZ",AZ,AZ1,1,dominant_region,"143,152.6994","239,935.4394",0.5966,59.6630,"143,152.6994",31.4931,5,1214,2078
3,04007,"Gila County, AZ",AZ,AZ2,1,dominant_region,"25,585.4229","25,990.6717",0.9844,98.4408,"25,585.4229",97.0599,3,304,309
4,04009,"Graham County, AZ",AZ,AZ4,1,dominant_region,"33,612.7114","60,980.8476",0.5512,55.1201,"33,612.7114",10.2402,2,355,621
5,04011,"Greenlee County, AZ",AZ,AZ4,1,dominant_region,"4,337.8142","4,337.8142",1.0000,100.0000,"4,337.8142",NaN,1,45,45
6,04012,"La Paz County, AZ",AZ,AZ1,1,dominant_region,"36,432.4339","39,248.6941",0.9282,92.8246,"36,432.4339",85.6492,2,377,404
7,04013,"Maricopa County, AZ",AZ,AZ2,1,dominant_region,"55,866.8124","60,036.9646",0.9305,93.0540,"55,866.8124",86.1081,2,730,781
8,04015,"Mohave County, AZ",AZ,AZ1,1,dominant_region,"87,845.1580","133,885.8976",0.6561,65.6120,"87,845.1580",31.3184,3,981,1447
9,04017,"Navajo County, AZ",AZ,AZ3,1,dominant_region,"168,893.0424","179,979.2473",0.9384,93.8403,"168,893.0424",89.8085,3,1411,1556


Final rows: 428
Final counties: 419
Final assigned buildable MW: 9,957,637.09


## Step 16: Export final CSV files

The notebook writes:

1. `county_region_assignments_5pct_rule.csv`  
   Final retained county-region assignments.

2. `county_assignment_summary_5pct_rule.csv`  
   One row per county with first-place, second-place, retained, and excluded capacity information.

3. `counties_assigned_to_two_regions_5pct_rule.csv`  
   Only counties whose first- and second-place regions are within 5 percentage points.

4. `all_county_region_rankings.csv`  
   All original county-region pairs with ranks and keep/exclude decisions.

5. `county_region_assignment_5pct_qa.csv`  
   Quality-assurance results.

6. `county_region_assignment_threshold_sensitivity.csv`  
   Sensitivity results for alternative percentage-point thresholds.

7. `county_region_assignment_5pct_summary.txt`  
   A concise human-readable summary.

In [77]:
FINAL_ASSIGNMENT_OUT = (
    OUT_DIR
    / "county_region_assignments_5pct_rule.csv"
)

COUNTY_SUMMARY_OUT = (
    OUT_DIR
    / "county_assignment_summary_5pct_rule.csv"
)

TWO_REGION_OUT = (
    OUT_DIR
    / "counties_assigned_to_two_regions_5pct_rule.csv"
)

ALL_RANKINGS_OUT = (
    OUT_DIR
    / "all_county_region_rankings.csv"
)

QA_OUT = (
    OUT_DIR
    / "county_region_assignment_5pct_qa.csv"
)

SENSITIVITY_OUT = (
    OUT_DIR
    / "county_region_assignment_threshold_sensitivity.csv"
)

CAPACITY_SUMMARY_OUT = (
    OUT_DIR
    / "county_region_assignment_capacity_summary.csv"
)

final_assignments.to_csv(
    FINAL_ASSIGNMENT_OUT,
    index=False,
)

county_assignment_summary.to_csv(
    COUNTY_SUMMARY_OUT,
    index=False,
)

two_region_assignments.to_csv(
    TWO_REGION_OUT,
    index=False,
)

ranked.to_csv(
    ALL_RANKINGS_OUT,
    index=False,
)

qa_table.to_csv(
    QA_OUT,
    index=False,
)

threshold_sensitivity.to_csv(
    SENSITIVITY_OUT,
    index=False,
)

capacity_summary.to_csv(
    CAPACITY_SUMMARY_OUT,
    index=False,
)

print("Wrote:", FINAL_ASSIGNMENT_OUT)
print("Wrote:", COUNTY_SUMMARY_OUT)
print("Wrote:", TWO_REGION_OUT)
print("Wrote:", ALL_RANKINGS_OUT)
print("Wrote:", QA_OUT)
print("Wrote:", SENSITIVITY_OUT)
print("Wrote:", CAPACITY_SUMMARY_OUT)

Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule/county_region_assignments_5pct_rule.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule/county_assignment_summary_5pct_rule.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule/counties_assigned_to_two_regions_5pct_rule.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule/all_county_region_rankings.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule/county_region_assignment_5pct_qa.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analysis/validation_outputs/county_region_assignment_5pct_rule/county_region_assignment_threshold_sensitivity.csv
Wrote: /Users/laurenvo/Documents/Github/solar-county-analy

## Step 17: Final summary

In [ ]:
summary_text = f"""
COUNTY-TO-REGION ASSIGNMENT USING THE 5-PERCENTAGE-POINT RULE

Input
-----
Input county-region rows: {len(county_region_clean):,}
Original unique counties: {original_county_count:,}
Original county-region pairs: {len(county_region_clean):,}

Assignment rule
---------------
- Keep the first-place model region for every county.
- Keep the second-place region when its capacity share is within
  {TOP_TWO_GAP_THRESHOLD_PCT:.1f} percentage points of first place.
- Exclude third-place and lower regions.
- Use each retained county-region pair's actual capacity_mw as
  assigned_buildable_mw.

Results
-------
Counties assigned to one region: {one_region_counties:,}
Counties assigned to two regions: {two_region_counties:,}
Total retained county-region assignments: {total_assignments:,}
All counties represented: {retained_county_count == original_county_count}

Capacity
--------
Original candidate capacity: {original_total_capacity_mw:,.2f} MW
Retained candidate capacity: {retained_total_capacity_mw:,.2f} MW
Retained capacity percentage: {retained_total_capacity_pct:.2f}%
Excluded candidate capacity: {excluded_total_capacity_mw:,.2f} MW
Excluded capacity percentage: {excluded_total_capacity_pct:.2f}%

Quality assurance
-----------------
QA checks passed: {int(qa_table["passed"].sum())} of {len(qa_table)}
All QA checks passed: {qa_table["passed"].all()}

Main output
-----------
{FINAL_ASSIGNMENT_OUT}
""".strip()

print(summary_text)

SUMMARY_OUT = (
    OUT_DIR
    / "county_region_assignment_5pct_summary.txt"
)

SUMMARY_OUT.write_text(summary_text)

print("\nWrote:", SUMMARY_OUT)

COUNTY-TO-REGION ASSIGNMENT USING THE 5-PERCENTAGE-POINT RULE

Input
-----
Input county-region rows: 739
Original unique counties: 419
Original county-region pairs: 739

Assignment rule
---------------
- Keep the first-place model region for every county.
- Keep the second-place region when its capacity share is within
  5.0 percentage points of first place.
- Exclude third-place and lower regions.
- Use each retained county-region pair's actual capacity_mw as
  assigned_buildable_mw.

Results
-------
Counties assigned to one region: 410
Counties assigned to two regions: 9
Total retained county-region assignments: 428
All counties represented: True

Capacity
--------
Original candidate capacity: 11,715,680.05 MW
Retained candidate capacity: 9,957,637.09 MW
Retained capacity percentage: 84.99%
Excluded candidate capacity: 1,758,042.96 MW
Excluded capacity percentage: 15.01%

Quality assurance
-----------------
QA checks passed: 7 of 7
All QA checks passed: True

Main output
-----------


: 

## Interpretation

The final table implements a reduced county-region structure:

- most counties appear once, in their dominant model region;
- counties with nearly tied first- and second-place capacity shares appear twice;
- each retained row uses only the candidate solar capacity actually associated with that region.

The output is a pre-assignment table. It can next be joined with the 1,000-cluster allocation workflow so cluster counts are distributed across the retained county-region assignments rather than all original county-region pairs.